# Atelier complémentaire B : la diffusion du téléphone mobile

Cours de M1 Économie de l'innovation, Université de Picardie Jules Verne

**La question de l'atelier.** Comment le téléphone mobile s'est diffusé dans des pays aux revenus et aux trajectoires différents, ce que mesurent les séries disponibles et ce qu'elles ne permettent pas de conclure.

| | |
|---|---|
| Niveau | Pour aller plus loin, à distance, après le TD 3 |
| Ce que l'atelier ajoute aux TD | télécharger des données par API et décrire une courbe de diffusion (cf. Chapitre 11) |
| Utile pour le projet | si votre question porte sur l'adoption d'une technologie |

Cet atelier est facultatif. Ne pas le faire n'est jamais pénalisé.

### Objectifs

- interroger l'API d'une source publique et documenter la source,
- distinguer le nombre d'abonnements du taux d'abonnements par habitant.e,
- décrire des trajectoires dans le temps avec des graphiques lisibles,
- relier diffusion, effets de réseau et infrastructures sans confondre description et causalité.

## Données et provenance

Les indicateurs viennent de l'API publique de la Banque mondiale (World Development Indicators). Les séries et leurs définitions peuvent être révisées. Consultez la page de chaque indicateur avant de réutiliser les données.

- `IT.CEL.SETS.P2` : abonnements mobiles pour 100 personnes,
- `IT.CEL.SETS` : nombre total d'abonnements mobiles,
- unité d'observation : le pays-année, de 2000 à 2024,
- API : https://api.worldbank.org/v2/

Un abonnement n'est pas une personne. Une même personne peut avoir plusieurs cartes SIM et un abonnement peut être partagé. Le taux pour 100 personnes peut donc dépasser 100.

Le notebook utilise `pandas` et `matplotlib`. Il faut une connexion Internet.

In [ ]:
import json
from urllib.error import HTTPError, URLError
from urllib.parse import urlencode
from urllib.request import urlopen

import pandas as pd
import matplotlib.pyplot as plt

COUNTRIES = ['FRA', 'DEU', 'USA', 'CHN', 'IND', 'BRA', 'ZAF']
START_YEAR = 2000
END_YEAR = 2024
plt.style.use('seaborn-v0_8-whitegrid')

## 1. Interroger l'API

La fonction ci-dessous lit les pages successives de l'API, conserve les valeurs manquantes et renvoie une table. Une erreur de réseau est signalée. Elle ne doit pas être remplacée par une série inventée. Il n'est pas nécessaire de comprendre chaque ligne de la fonction pour l'utiliser.

In [ ]:
def fetch_world_bank_indicator(indicator, start_year=START_YEAR, end_year=END_YEAR):
    base_url = f'https://api.worldbank.org/v2/country/all/indicator/{indicator}'
    rows = []
    page = 1
    total_pages = 1

    while page <= total_pages:
        query = urlencode({
            'format': 'json',
            'date': f'{start_year}:{end_year}',
            'per_page': 1000,
            'page': page,
        })
        with urlopen(f'{base_url}?{query}', timeout=30) as response:
            payload = json.loads(response.read().decode('utf-8'))

        if not isinstance(payload, list) or len(payload) < 2:
            raise ValueError(f'Réponse API inattendue pour {indicator}: {payload}')

        metadata, records = payload[0], payload[1]
        total_pages = int(metadata.get('pages') or 1)
        for record in records or []:
            country = record.get('country') or {}
            rows.append({
                'country_code': record.get('countryiso3code'),
                'country': country.get('value'),
                'year': int(record['date']),
                'value': record.get('value'),
                'indicator': indicator,
            })
        page += 1

    return pd.DataFrame(rows)

In [ ]:
try:
    mobile_rate = fetch_world_bank_indicator('IT.CEL.SETS.P2')
    mobile_total = fetch_world_bank_indicator('IT.CEL.SETS')
    print(f'Taux : {len(mobile_rate):,} lignes ; total : {len(mobile_total):,} lignes')
except (OSError, ValueError, KeyError) as error:
    mobile_rate = pd.DataFrame()
    mobile_total = pd.DataFrame()
    print('Données non récupérées. Vérifiez la connexion ou réessayez plus tard.')
    print(f'Détail technique : {error}')

## 2. Inspecter et contrôler la base

Avant le graphique, vérifiez les dimensions, le type des colonnes, les années, les doublons et les valeurs manquantes. L'API renvoie aussi des groupes de pays (zones, niveaux de revenu). On ne garde ici que les pays de la liste.

In [ ]:
if not mobile_rate.empty:
    rate = mobile_rate.loc[mobile_rate['country_code'].isin(COUNTRIES)].copy()
    rate['value'] = pd.to_numeric(rate['value'], errors='coerce')
    print('Dimensions :', rate.shape)
    print('Années :', rate['year'].min(), 'à', rate['year'].max())
    print('Doublons pays-année :', rate.duplicated(['country_code', 'year']).sum())
    display(rate.isna().sum().rename('valeurs manquantes').to_frame())
    display(rate.head())
else:
    print('La table est vide : réexécutez la cellule API lorsque la connexion est disponible.')

## 3. Décrire les trajectoires

### Tâche A : taux pour 100 personnes

Le taux facilite les comparaisons mais ne représente pas une part de personnes. Complétez la figure : une légende, une unité, une période et une interprétation de deux phrases.

In [ ]:
if not mobile_rate.empty:
    rate = mobile_rate.loc[mobile_rate['country_code'].isin(COUNTRIES)].copy()
    rate['value'] = pd.to_numeric(rate['value'], errors='coerce')
    pivot_rate = rate.pivot(index='year', columns='country', values='value')
    ax = pivot_rate.plot(figsize=(11, 6), linewidth=2)
    ax.set(title='Abonnements mobiles pour 100 personnes',
           xlabel='Année', ylabel='Abonnements pour 100 personnes')
    ax.legend(title='Pays', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.tight_layout()
    plt.show()
else:
    print('Figure non produite : aucune donnée disponible.')

**À vous.** Repérez les trajectoires qui augmentent le plus vite et les différences de niveau de départ ou de calendrier. Dites si une courbe qui s'aplatit signifie forcément que l'adoption s'arrête. Citez une limite de l'indicateur.

*Votre réponse :*

### Tâche B : niveaux et dénominateurs

Comparez le nombre total d'abonnements (`IT.CEL.SETS`) au taux pour 100 personnes. Les pays les plus peuplés dominent les niveaux. Expliquez à quelle question répond chacune des deux mesures.

In [ ]:
if not mobile_total.empty:
    totals = mobile_total.loc[mobile_total['country_code'].isin(COUNTRIES)].copy()
    totals['value'] = pd.to_numeric(totals['value'], errors='coerce')
    latest_year = int(totals['year'].max())
    latest = totals.loc[totals['year'].eq(latest_year)].sort_values('value', ascending=False)
    ax = latest.plot.bar(x='country', y='value', legend=False, figsize=(9, 4), color='#00686d')
    ax.set(title=f'Abonnements mobiles, niveau absolu ({latest_year})',
           xlabel='Pays', ylabel='Nombre d’abonnements')
    plt.tight_layout()
    plt.show()
else:
    print('Figure non produite : aucune donnée disponible.')

## 4. Relier les données au cours

**À vous.** Répondez en quelques lignes à chaque point :

1. la différence entre invention, innovation et diffusion que ce cas illustre (cf. Chapitre 2),
2. le rôle possible des infrastructures, du prix des appareils, des revenus et des effets de réseau dans l'adoption,
3. ce qu'une hausse des abonnements permet de dire d'un effet du mobile sur la productivité ou l'inclusion financière et ce qu'elle ne permet pas de dire,
4. trois variables supplémentaires et une démarche qui permettraient d'étudier un effet causal,
5. les conditions pour comparer des trajectoires de pays différents : revenu, démographie, couverture des réseaux, définitions statistiques.

## 5. Prolongement : diffusion et R&D

La R&D est un intrant de l'innovation. Les abonnements décrivent sa diffusion. Une corrélation positive entre les deux n'établit pas que la R&D a causé l'adoption. Le revenu, les infrastructures, les politiques publiques et les tendances communes agissent sur les deux séries.

Téléchargez l'indicateur `GB.XPD.RSDV.GD.ZS` (R&D en % du PIB), examinez les valeurs manquantes et comparez seulement les pays-années communs. Présentez le résultat comme une association.

## Limites et reproductibilité

- L'API peut être indisponible ou réviser les observations. Notez la date d'extraction dans votre rendu.
- Un abonnement ne mesure pas le nombre de personnes équipées. Il ne dit rien des usages, de la couverture du réseau ou de la qualité du service.
- Les écarts entre pays peuvent refléter le revenu, la démographie et les infrastructures.
- Un graphique et une corrélation ne démontrent pas un effet causal.
- Ne remplacez pas une valeur absente par zéro sans justification. Documentez toute exclusion ou transformation.

## Vérifications avant remise

- [ ] Le notebook s'exécute de haut en bas sans erreur.
- [ ] La question, l'unité d'observation, la période et les sources sont explicites.
- [ ] Chaque graphique a un titre, des axes, une unité et une interprétation économique.
- [ ] Les niveaux sont distingués des taux rapportés à la population.
- [ ] Les valeurs manquantes et les limites sont discutées.
- [ ] Aucune conclusion causale n'est tirée d'une tendance ou d'une corrélation.
- [ ] Les références et la date d'accès aux données sont indiquées.

### Références

- Banque mondiale, World Development Indicators : https://data.worldbank.org/
- OCDE/Eurostat (2018), *Oslo Manual 2018: Guidelines for Collecting, Reporting and Using Data on Innovation*.
- Rogers, E. M. (2003), *Diffusion of Innovations*, 5e éd., Free Press.